## Problem 1 - Discrete Mathematics: Shareholding Networks *(compulsory, 25 points)*
A business-information company stores shareholding data for $n$ legal entities as a directed graph $G = (V,E)$, where an edge $(u, v)$ means company $u$ directly holds shares in company $v$.

1. **(27%)** Define "circular shareholding" in graph-theoretic terms. Then prove that $G$ has no circular shareholding if and only if the companies can be ordered so that every edge points from an earlier company to a later one (a topological ordering). Prove both directions.

Circular shareholding คือรูปแบบการถือหุ้นไขว้ระหว่างบริษัทตั้งแต่ 3 บริษัทขึ้นไปเป็นวงกลม\
ในเชิงทฤษฎีกราฟ ถ้าเกิดแต่ละบริษัทเป็น node และ การถือหุ้นจะทำให้เกิด edge ที่แสดงการถือหุ้น เช่นถ้า บริษัท A ถือหุ้น บริษัท B จะทำให้เกิด A -> B ซึ่ง circular shareholding จะเกิดขึ้นก็ต่อเมื่อมีการถือหุ้นกันเป็นวงกลม (เมื่อพยายามเดินไปตามทาง edge จะมีทิศทางที่กลับมาจุดเริ่มต้นได้) เช่น เมื่อ A -> B, B -> C และ C -> A

***พิสูจน์***\
**ถ้า G ไม่มี circular shareholding แล้ว บริษัทจะสามารถเรียงเป็นเส้นตรง และ edge ชี้ไปทางเดียวกันได้** 
เมื่อไม่มี circular shareholding หมายถึงว่าต้องมีอย่างน้อย 1 บริษัทที่ไม่มีบริษัทอื่นถือหุ้นอยู่เสมอ ดังนั้น เมื่อลองทำการตัดบริษัทที่ไม่มีบริษัทอื่นถือหุ้นอยู่ออกไป คุณสมบัติเดิมของการไม่มี circular shareholding จะยังคงอยู่เสมอ

**ถ้า บริษัทจะสามารถเรียงเป็นเส้นตรง และ edge ชี้ไปทางเดียวกันได้ แล้ว G ไม่มี circular shareholding** 
การที่สามารถเรียงไปทางเดียวกันได้โดยไม่มี edge เส้นไหนชี้ย้อนกลับ แสดงให้เห็นว่าไม่มีบริษัทไหนที่มีการถือหุ้นย้อนกลับมาที่บริษัทก่อนหน้าได้ แสดงให้เห็นว่า circular shareholding จะไม่สามารถเกิดขึ้นได้

2. **(33%)** Design an algorithm that returns every group of companies involved in circular shareholding with one another. State its time complexity in Big-O of $|V|$ and $|E|$ with justification. Implement it in Python and test it on a random graph with 100,000 nodes. Do not use ready-made functions from graph libraries (you may use them to verify your answer).

Algorithm ใช้หลักการที่ว่า ถ้าเป็น circular ไม่ว่าจะเดินไปตามกราฟหรือย้อนศรกราฟจะสามารถเดินวนได้เหมือนกัน\
ดังนั้นในขั้นแรกจะทำการสร้าง directed กราฟของแต่ละ node และ กราฟที่เกิดจากการ reverse edge\
จากนั้นทางการ DFS เพื่อหาลำดับของการเจอในกราฟ โดย node ที่ขึ้นมาก่อนบน list คืออยู่ปลายกราฟ และ node ที่อยู่สุดท้ายคือ node ที่มีแนวโน้ม เป็น root ที่สุด\
หลักจากนั้นจึงลองทำการตรวจสอบกราฟที่เป็น reverse โดยเริ่มจาก node สุดท้ายที่มีแนวโน้มเป็น root ที่สุด เพื่อตรวจสอบว่าสามารถเดินทางได้ไหม ถ้าได้แสดงว่าเกิด circular ขึ้น ถ้าเกิด dead end แสดงว่า node นั้นไม่ได้เป็น circular\
(ได้มีการตรวจสอบ algorithm ที่ src/q1/test_shareholding.py)

In [8]:
def circular_shareholding(n, edges):
  # Build the graph and the reversed graph
  adj = [[] for _ in range(n)]
  rev_adj = [[] for _ in range(n)]
    
  for u, v in edges:
    adj[u].append(v)
    rev_adj[v].append(u)
        
  # DFS to find last order traversal
  visited = [False] * n
  finished = [False] * n
  order = []
    
  for i in range(n):
    if not visited[i]:
      stack = [i]
      while stack:
        curr = stack[-1]
        if not visited[curr]:
          visited[curr] = True
          # Push unvisited neighbors to stack
          for neighbor in adj[curr]:
            if not visited[neighbor]:
              stack.append(neighbor)
        else:
          curr = stack.pop()
          # Only append to order when finished with all children
          if not finished[curr]:
            finished[curr] = True
            order.append(curr)
                        
  # Find circular groups using the reversed graph
  visited_rev = [False] * n
  circular_groups = []

  # Process in reverse of finishing times
  for i in reversed(order):
    if not visited_rev[i]:
      stack = [i]
      visited_rev[i] = True
      current_scc = []
      
      while stack:
        curr = stack.pop()
        current_scc.append(curr)
        
        for neighbor in rev_adj[curr]:
          if not visited_rev[neighbor]:
            visited_rev[neighbor] = True
            stack.append(neighbor)
        
        # > 1 companies in the SCC
        if len(current_scc) > 1:
          if current_scc not in circular_groups:
            circular_groups.append(current_scc)

  return circular_groups

In [9]:
import random

V = 100000
E = 250000

print("Generating graph")
edges = [(random.randint(0, V - 1), random.randint(0, V - 1)) for _ in range(E)]

circular_groups = circular_shareholding(V, edges)
print(f"New algorithm")
print(f"Found {len(circular_groups)} distinct circular shareholding groups.")
# print(circular_groups)


Generating graph
New algorithm
Found 1 distinct circular shareholding groups.


3. **(40%)** A registry holds $n=2,000,000$ company records, and you must run entity resolution to find duplicates.

a. If every pair is compared, how many comparisons are needed?

**ประมาณ 1,999,999,000,000 ครั้ง**\
การเปรียบเทียบสามารถใช้สูตรการจัดกลุ่มทีละ 2 ได้ (คำนวณที่ src/q1/combination.py)\
สูตรคำนวณคือ $$TotalComparisons = \frac{n*(n - 1)}{2}$$ 

b. If records are split into $k$ equal-sized blocks (blocking) and compared only within blocks, how many comparisons are needed? Give a formula in $n$ and $k$, then evaluate it for $k=1,000$.

**ประมาณ 1,999,000,000 ครั้ง**\
โดยใน 1 กลุ่ม จะมี 2,000 บริษัท ทำให้เกิดการจับคู่ในกลุ่มทั้งหมด 1,999,000 ครั้ง ทำให้ทั้งหมดต้องจับคู่ 1,999,000,000 ครั้ง (คำนวณที่ src/q1/combination.py)\
สูตรคำนวณโดยคือ $$TotalComparisons = k*\frac{n/k*(n/k - 1)}{2} = \frac{n*(n/k - 1)}{2}$$ 

c. Prove that when block sizes are unequal, the number of comparisons is always greater than or equal to the equal-size case (hint: convexity).

ลักษณะของกราฟ จากสูตร $f(x) = \frac{x*(x - 1)}{2}$ ในข้อ 3a เป็นลักษณะโค้งหงายขึ้น ซึ่งรู้ได้จากการที่ $$f''(x) = 1$$ ซึ่งมากกว่า 0 และจาก Jensen's Inequality ได้มีการบอกไว้ว่า $$f(\sum_{i=1}^{n}w_{i}x_{i}) ≤ \sum_{i=1}^{n}{w_{i}f(x_{i})}$$ ซึ่งหมายความว่า ค่าของฟังก์ชันที่ได้จากค่าเฉลี่ยถ่วงน้ำหนักของจุดต่าง ๆ จะน้อยกว่าหรือเท่ากับ ค่าเฉลี่ยถ่วงน้ำหนักของค่าฟังก์ชันที่นั้น ๆ เสมอ
ซึ่งหมายความว่าผลรวมของการที่ block เท่ากัน จะมากกว่าหรือเท่ากับการที่จัด block ที่ไม่เท่ากันเสมอ

เช่นในการแบ่งกลุ่ม 2 กลุ่ม\
ถ้าแบ่ง 1,000,000 2 กลุ่มเท่ากัน จะมีการเปรียบเทียบ  999,999,000,000 ครั้ง\
ถ้าแบ่งเป็น 2 กลุ่มที่ไม่เท่ากัน ที่ 900,000 และ 1,100,000 จะมีการเปรียบเทียบ 1,009,999,000,000 ครั้ง\
ซึ่งมีจำนวนมากกว่า (คำนวณที่ src/q1/combination.py)

d. Explain which kinds of duplicate pairs blocking can miss, and how you would mitigate this.

จะทำให้หาค่าซ้ำใน กรณีที่ค่าที่ซ้ำกับอยู่กันคนละ block ไม่เจอ เนื่องจากเป็นแค่การเปรียบเทียบแค่ภายใน block 
โดยเราจะสามารถแก้ปัญหานี้ได้ 2 วิธี
1. ทำการแบ่งว่าแต่ละ block เป็น block สำหรับอะไร ดังนั้นค่าประเภทเดียวกันก็จะอยู่ใน block เดียวกัน ซึ่งหมายถึงค่าที่เหมือนกันจะจะอยู่ใน block เดียวกันอย่างแน่นอน
2. ทำการ sort ข้อมูลตั้งแต่ก่อนใส่ค่าเพื่อให้ข้อมูลทุกอย่างเรียงต่อกัน และในการเปรียบเทียบให้เพื่อการเปรียบเทียบในค่าสุดท้ายของ block กับค่าแรกใน block ถัดไปเพื่อตรวจสอบว่าค่านั้นมีหรือยัง

ซึ่งทั้ง 2 วิธีมีข้อดีข้อเสียต่างกันข้อ 1 จำเป็นต้องการเกณฑ์ในการแบ่งแยกแต่งละ block และแต่ละ block จะมีความเป็นไปได้สูงที่จะมีขนาดที่ไม่เท่ากัน ซึ่งหมายถึงการเปรียบเทียบจำนวณครั้งก็จะมากขึ้นตามที่ได้มีการพิสูจน์ไปในข้อ 3c
ส่วนวิธีที่ 2 เมื่อมีการเพิ่มเติมข้อมูลมาใหม่ต้องมีการตรวจสอบข้อมูลกับ block ต่าง ๆ อยู่เสมอ และควรจะต้องทำการจัดระเบียบให้ดี เพื่อป้องกันข้อผิดพลาดที่จะใส่ข้อมูลผิด block ซึ่งจะทำให้เกิดปัญหาในระยะยาวเมื่อมีการใส่ข้อมูลผิดพลาด หรือรวมถึงอาจจะต้องมีการย้ายข้อมูลระหว่าง block ในกรณีที่ต้องการให้ข้อมูลแต่ละ block มีขนาดเท่ากัน

## Problem 2 - Linear Algebra: Indirect Ownership and PCA *(compulsory, 25 points)*

### Part A: Indirect ownership
Let $M$ be the direct-ownership matrix, where $M_{ij}$ is the fraction of company $j$ held directly by company $i$. For companies A, B, C and D:

| Holder \ Company held | A | B | C | D |
| :--- | :--- | :--- | :--- | :--- |
| **A** | 0 | 0.6 | 0 | 0.1 |
| **B** | 0 | 0 | 0.5 | 0 |
| **C** | 0.2 | 0 | 0 | 0.4 |
| **D** | 0 | 0 | 0 | 0 |

Define total ownership (direct + indirect through every chain) as:
$$T = M + M^{2} + M^{3} + \dots$$

1. **(20%)** Prove that if the series converges, $T=(I-M)^{-1}-I$, and state the convergence condition in terms of the spectral radius. Explain why real-world ownership matrices usually satisfy this condition.

พิสูจน์ว่า
$$(I-M)^{-1}-I=M + M^{2} + M^{3} + \dots$$

ตามกฎของ Neumann series บอกไว้ว่าเมื่อ T < 1 
$$(I-T)^{-1}=I + T + T^{2} + T^{3} + \dots$$

ดังนั้นในสมการเมื่อ M < 1
$$(I-M)^{-1}=I + M + M^{2} + M^{3} + \dots$$
$$(I-M)^{-1}-I=I + M + M^{2} + M^{3} + \dots - I$$
$$(I-M)^{-1}-I=M + M^{2} + M^{3} + \dots$$

และในกรณีที่ M ≥ 1 จำทำให้ค่า $T = M + M^{2} + M^{3} + \dots$ เป็น infinity ซึ่งในสถานการณ์ของการถือหุ้น ค่อนข้างมีน้อยที่ผู้ถือหุ้นหลักจะมีการถือหุ้นแบบ 100% (M = 1) เนื่องจากมักจะมีผู้ถือหุ้นรายย่อยมามีส่วนร่วมอยู่เสมอ

2. **(27%)** Compute $T$ for the table above, both by hand (show how the structure of the matrix simplifies the calculation) and with NumPy. What is A's total percentage ownership of D?

**By Hand**\
A -> D ownership:
- A -> D : 0.1
- A -> B -> C -> D : 0.6 * 0.5 * 0.4 = 0.12\
Total = 0.1 + 0.12 = 0.22\
In percentage = 22%

**By NumPy**

In [10]:
import numpy as np

def compute_t(m):
  I = np.linalg.matrix_power(m, 0)
  M_inv = np.linalg.inv(I - m)
  T = M_inv - I
  return T

ownership_matrix = np.array([[0, 0.6, 0, 0.1],
                            [0, 0, 0.5, 0],
                            [0.2, 0, 0, 0.4],
                            [0, 0, 0, 0]])
T = compute_t(ownership_matrix)
print("A ownership D: ", T[0, 3])
print(f"In percentage: {T[0, 3] * 100:.2f}%")

A ownership D:  0.2340425531914894
In percentage: 23.40%


3. **(13%)** The diagonal entries of $T$ for A, B and C are non-zero. What does this mean in business terms, and how should it be reported to business stakeholders?

หมายความว่าบริษัทมีการถือหุ้นของบริษัทตัวเองในทางอ้อมอยู่ทำให้เกิด circular ownership ทำให้เงินทุนรวมอาจจะสูงเกินจริงในทางบัญชีจึงต้องมีการชี้แจ้งแล้วหักค่านี้ออกเมื่อคำนวณมูลค่าจริงของบริษัทจากบุคคลภายนอก

### Part B: PCA and SVD
Let $X$ be a centred $n \times p$ data matrix.

1. **(20%)** Prove that the unit vector $w$ that maximises the variance of $Xw$ is the eigenvector of the covariance matrix with the largest eigenvalue (use a Lagrange multiplier).

สามารถหา Covariance Matrix ได้จาก $\text{Var}(x) = \frac{1}{n} (x_c)^T (x_c)$ และ $C = \frac{1}{n} X^T X$ ดังนั้น
$$\text{Var}(Xw) = \frac{1}{n} (Xw)^T (Xw) = \frac{1}{n} w^T X^T X w = w^T C w$$
ถ้า maximize $w$ และเป็น unit vector $w^T w = 1$
Lagrangian function บอกว่า $\mathcal{L}(x, y, \lambda) = f(x, y) - \lambda(g(x, y) - c)$
จะได้ว่า $$\mathcal{L}(w, \lambda) = w^T C w - \lambda (w^T w - 1)$$
หา maximum จากการกำหนด gradient ของ Lagrangian ของ $w$ เป็น 0 จะได้ว่า
$$\nabla_w \mathcal{L}(w, \lambda) = 2Cw - 2\lambda w = 0$$
$$Cw = \lambda w$$
และเมื่อลองนำไปแทนค่าในสมการ $\text{Var}(Xw) = w^T C w$ จะได้ว่า
$$\text{Var}(Xw) = w^T C w = w^T \lambda w = \lambda (w^T w) = \lambda$$

2. **(13%)** Show the relationship between the principal components and the SVD $X=U\Sigma V^{T}$, and explain why in practice PCA should be computed via the SVD rather than by an eigen-decomposition of $X^{T}X$.

จากสูตรของ SVD $X=U\Sigma V^{T}$ ถ้าทำการแทนค่า $X^{T}X$ เข้าไปในสมการจะได้ว่า
$$X^T X = (U \Sigma V^T)^T (U \Sigma V^T) = U^T \Sigma V U \Sigma V^T$$
และเมื่อ $U$ เป็น orthogonal matrix จะทำให้ $U^T U = I$ ส่งผลให้
$$X^T X = V \Sigma^2 V^T$$
ซึ่งมีรูปแบบเหมือนสูตรของ eigen-decomposition ที่ $X^T X = Q \Lambda Q^T$\
ซึ่งหลายครั้งมีการเลือกใช้ SVD แทน PCA เนื่องจากว่า\
ในการคำนวณค่า $X^T X$ ผ่าน PCA นั้น จะทำให้ค่า Condition Number ของข้อมูลเพิ่มขึ้นเป็นกำลังสอง ซึ่งเมื่อเลขมีขนดใหญ่มาก ๆ จะทำให้ความแม่นยำลดลง\
นอกจากนี้การคำนวณ PCA มีการใช้เวลาและหน่อยความจำในการคำนวณค่อนข้างมาก โดยเฉพาะเมื่อ p มีขนาดใหญ่ ซึ่งในหลาย ๆ ครั้ง เราก็ไม่จำเป็นต้องใช้ทั้ง matrix ทำให้มีการคำนวณที่เกินความจำเป็น

3. **(7%)** When must variables be standardised before PCA? Give an example using company financial statement data

เมื่อข้อมูลมีหน่วยที่ต่างกัน หรือว่าช่วงของข้อมูลมีความต่างกันมาก เพราะ PCA ใช้เพื่อหาแกนที่มีความแปรปรวนสูงสุด จึงควรมีการทำ standardised เพื่อนหาความแปรแรวนจากขนาดของตัวเลขจริง ๆ
เช่น ในงบการเงินถ้าข้อมูลที่ได้มามีหน่วยที่ต่างกัน เช่น 
- ข้อมูลรายได้ 45,000,000,000 บาท
- อัตราส่วนหนี้สินต่อทุน 1.2 
- อัตรากำไร 8%

ซึ่งในกรณีนี้ถ้าไม่ standardised ก่อน ความแปรปรวนของข้อมูลรายได้จะสูงมาก ทำให้ PCA สนใจเพียงแค่ข้อมูลรายได้เพียงแค่อย่างเดียวโดยไม่สนตัวแปรอื่น ๆ ที่อาจจะทำให้ผลลัพธ์เปลี่ยนไป

## Problem 3 - Probability & Statistics: Data-driven Decisions *(elective, 50 points)*

### Part A: Bayes and fraud detection
A fraud-detection system has 95% sensitivity and 98% specificity, while only 1% of transactions are actually fraudulent.

1. **(10%)** When the system raises an alert, what is the probability that the transaction is truly fraudulent?

**ประมาณ 1.03%**
โดยคำนวณจาก Bayes Theorem $$P(Fraud|Flagged) = \frac{P(Flagged|Fraud)*P(Fraud)}{P(Flagged)}$$ จะได้ว่าความเป็นไปได้ที่ transaction จะ fraudulent จริง ๆ อยู่ที่ $\frac{0.95*0.01}{(0.95*0.01)+((1-0.98)*0.99)}=0.3242$ หรือ 32.42%

2. **(10%)** With 1,000,000 transactions per month and an investigation team that can review 10,000 transactions per month, is this system workable in practice? Support your answer with numbers.

จากโจทย์ทราบว่า
- True Positive Rate = 0.95
- True Negative Rate = 0.98
- Negative = 0.99
- Positive = 0.01

ทำให้สามารถรู้ได้ว่า
- True Positive = $0.95 * 0.01 * 1000000 = 9500$
- True Negative = $0.98 * 0.99 * 1000000 = 970200$
- False Positive = $0.02 * 0.99 * 1000000 = 19800$
- False Negative = $0.05 * 0.01 * 1000000 = 500$
- Precision = 32.42% (จากข้อ3A.1)

ยังไม่เหมาะกับการใช้งานจริง เนื่องจากตัวเลขข้างต้นจะทำให้เห็นว่าวิธีนี้มีความแม่นยำค่อนข้างต่ำอยู่ที่ 32.42%\
ส่งผลให้ระบบมีการแจ้งเตือนบ่อยมากถึง 29,300 ครั้ง ซึ่งในจำนวนนี้มีที่เป็น transaction ที่หลอกลวงจริงแค่ 9,500 ทำให้ยากต่อการที่จะพบ transaction ที่มีปัญหาทั้งหมดจากในกลุ่มที่มีการแจ้งเตือนได้

3. **(10%)** Propose at least two ways to improve the process (e.g. cost-based threshold selection, or multi-stage review) and quantify the impact of at least one.

จากข้อ 3A.2 แสดงให้เห็นว่าปัญหาใหญ่ที่ทำให้วิธีนี้ไม่เหมาะอันดับ 1 อยู่ที่ False Positive ที่มีปริมาณค่อนข้างมาก จึงทำให้ยากต่อการตรวจสอบ ดังนั้นจึงควรที่จะลดค่าที่ทำให้แสดงออกมาว่า transaction หลอกลอง โดยวิธีที่เป็นไปได้ในการลดปัญหานี้คือ
1. ในเรื่องของเกณฑ์การวัดอาจจะควรทำให้ละเอียดขึ้น โดยมีการดูค่าอื่น ๆ เพิ่มเติมเข้าไป หรือ เพิ่มเกณฑ์ให้มากขึ้น\
   แต่วิธีนี้จะส่งผลให้ช่องโหว่ที่ทำให้เกิด False Positive มีมากขึ้นแทนถ้าเกณฑ์มีความละเอียดมากเกินไป ซึ่งโดยทั้วไปการเกิด False Negative เป็นปัญหาใหญ่กว่า False Positive อยู่มากจึงอาจจะทำเกิดปัญหาที่มากกว่าเดิมแทน
2. คงวิธีเดิมไว้ แต่หลักจากที่มีผลออกมาว่าเป็น positive แล้วทำการตรวจสอบอัตโนมัติเพิ่มเติมผ่านเงื่อนไขอื่นที่มีความซับซ้อนเพิ่มมากขึ้นแทน เช่น ในเรื่องของ ประวัติการทำธุรกรรม\
   แต่วิธีนี้ถึงแม้จะช่วยลด False Positive ได้จากการตรวจสอบซ้ำ แต่การตรวจสอบซ้ำจำนวนมากก็จะมีการกิน resource ต่าง ๆ เพิ่มขึ้นได้ นอกจากนี้ถ้าเป็นการตรวจสอบผ่านประวัติการทำธุรกรรมอาาจจะพบปัญหาในเกณฑ์การตรวจสอบบัญชีใหม่ที่ยังไม่มีประวัติมาก่อนได้

### Part B: A/B testing
A team tests a new sign-up page with 5,000 users per group. The control group converts at 4.0% and the treatment group at 4.6%.

1. **(15%)** Test the hypothesis at $\alpha=0.05$. Report the test statistic, the p-value and a 95% confidence interval for the difference.

จากผลการวิเคราะห์ข้อมูล พบว่า treatment group ไม่มีผลต่อ control group อย่างมีนัยสำคัญทางสถิติที่ระดับ p < 0.05 (z = 1.4789, p = 0.1392) จึงปฏิเสธสมมติฐานที่ในการเปลี่ยนแปลง sign-up page

ซึ่งคำนวณได้จากการคำนวณหา z-score ของโจทย์ จากสูตรนี้ (คำนวณที่ ./src/q3/hypothesis.py)
$$\hat{p} = \frac{x_1 + x_2}{n_1 + n_2}$$
$$SE_{\text{pooled}} = \sqrt{\hat{p}(1 - \hat{p})\left(\frac{1}{n_1} + \frac{1}{n_2}\right)}$$
$$z = \frac{x_1 - x_2}{SE_{\text{pooled}}}$$

2. **(15%)** Find the per-group sample size needed to detect a change from 4.0% to 4.6% with 80% power, and explain what the result of question 1 tells you about the design of this experiment.

ขนาด sample size โดยประมาณจะอยู่ที่ 17,940 คน โดยจากข้อ 1 จะเห็นว่าผลลัพธ์ที่ได้คือไม่มีความต่างอย่างมีนัยสำคัญจึงไม่สามารถสรุปผลที่ชัดเจนได้ ดังนั้นควรมีการเพิ่มขนาดจำนวนคำที่ทำการทดลองเพื่อให้เห็นแนวโน้มที่ชัดเจนยิ่งขึ้น

โดยสามารถหาขนาด sample size ได้จากสูตร คำนวณที่ src/q3/find_sample
$$n = \frac{(Z_{\alpha/2} + Z_{\beta})^2 \cdot [p_1(1-p_1) + p_2(1-p_2)]}{(p_2-p_1)^2}$$

3. **(10%)** A manager checks the results every day and stops the experiment as soon as $p<0.05$. Write a simulation (A/A tests, at least 1,000 runs) showing what the actual false-positive rate is.

จาก code เป็นการสมมติ user กลุ่มละ 5000 คน แบ่งใช้งาน 7 วัน

In [1]:
import numpy as np
from scipy.stats import norm

def calculate_p_value(n1, x1, n2, x2):

  if n1 <= 0 or n2 <= 0:
    print("No users")
    return None, None
  
  p_pool = (n1 * x1 + n2 * x2) / (n1 + n2)

  se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))

  z_score = (x2 - x1) / se

  p_value = 2 * (1 - norm.cdf(abs(z_score)))
    
  return z_score, p_value

In [2]:
simulations = 1000
days = 7
users = 5000
users_per_day = users // days
x = 0.04
alpha = 0.05
false_positives = 0

np.random.seed(1)

for _ in range(simulations):

  control_users = np.random.binomial(1, x, users)
  treatment_users = np.random.binomial(1, x, users)

  for day in range(1, days + 1):
    sum_user = day * users_per_day

    x1 = control_users[:sum_user].sum()
    x2 = treatment_users[:sum_user].sum()

    n1, x1 = sum_user, x1/sum_user
    n2, x2 = sum_user, x2/sum_user
    z_score, p_value = calculate_p_value(n1, x1, n2, x2)
          
    if p_value < alpha:
        false_positives += 1
        break

actual_fpr = (false_positives / simulations) * 100
print(f"False Positive Rate: {actual_fpr:.1f}%")


False Positive Rate: 18.3%


4. **(10%)** The team measured 20 metrics at once and found one with $p=0.03$. What should they conclude? Explain using Bonferroni and Benjamini-Hochberg.

ถ้าเป็นการเปรียบเทียบแค่ 2 เงื่อนไขอาจจะสามารถสรุปผลได้เลย แต่ในกรณีนี้ที่มีการตรวจสอบหลายเงื่อนไขพร้อมกันจำเป็นต้องทำการปรับเปลี่ยนเกณฑ์ที่ใช้ในการตัดสินใจ

**Bonferroni**\
สำหรับ Bonferroni จะใช้สูตร $\alpha'=\frac{\alpha}{m}$ ในการตัดสินใจ
โดยถ้า $\alpha = 0.05$ จะได้ว่า
$$\alpha'=\frac{0.05}{20}=0.0025$$
ซึ่งในที่นี้หลักจากปรับเกณฑ์แล้วแสดงให้เห็นว่าไม่มีนัยสำคัญระหว่างเงื่อนไขที่ตรวจสอบ

**Benjamini-Hochberg**\
สำหรับ Benjamini-Hochberg จะใช้สูตร $\alpha'=\frac{i}{m}\alpha$ ในการตัดสินใจ
โดยถ้า $\alpha = 0.05$ และเป็นค่าที่ต่ำที่สุดเป็นอันดับที่ 2 ที่คำนวณออกมาได้ จะได้ว่า
$$\alpha'=\frac{2}{20}0.05 = 0.005$$
ซึ่งในที่นี้หลักจากปรับเกณฑ์แล้วแสดงให้เห็นว่าไม่มีนัยสำคัญระหว่างเงื่อนไขที่ตรวจสอบ
และจะมีนัยสำคัญเมื่อเป็นอันดับที่ 12

ซึ่งจาก 2 วิธีนี้จะแสดงให้เห็นว่าการที่คำนวณได้ p=0.03 จากการเปรียบเทียบหลายเงื่อนไข และ $\alpha = 0.05$ จำเป็นต้องมีการเปลี่ยนเกณฑ์ซึ่งถ้าใช้วิธี Bonferroni จะสามารถสรุปได่เลยว่าไม่มีความต่างอย่างมีนัยสำคัญ ส่วนวิธี Benjamini-Hochberg จำเป็นต้องมีข้อมูลของลำดับผลลัพธ์ในการตรวจสอบด้วย โดยผลจะมีนัยสำคัญเมื่อข้อมูลนี้อยู่ลำดับที่ 12

### Part C: Conceptual understanding
**(20%)** Explain Simpson's paradox with a numerical example of your own (e.g. loan-approval rates broken down by business size), and describe how you would decide whether to look at the aggregated result or the per-group results.

เป็นปรากฎการณ์ที่เมื่อลองมองผลการทดสอบแบบแยกย่อยไปตามกลุ่มเห็นผลลัพธ์ไปในแนวทางหนึ่ง แต่เมื่อลองมองเป็นภาพรวมของจากข้อมูลทั้งหมดผลลัพธ์อาจจะไม่ได้แตกต่างอย่างมีนัยสำคัญหรือบางครั้งผลลัพธ์อาจจะกลับผลลัพธ์เป็นอีกทางได้เลย\
เช่น การในบริษัทกู้เงิน เมื่อเปรียบเทียบ 2 ธนาคาร
- ในบริษัทขนาดเล็ก ธนาคาร A อาจจะมีอัตราการให้กู้อยู่ที่ 15% ส่วน ธนาคาร B มีอัตราการให้กู้อยู่ที่ 20%
- ในบริษัทขนาดใหญ่ ธนาคาร A อาจจะมีอัตราการให้กู้อยู่ที่ 75% ส่วน ธนาคาร B มีอัตราการให้กู้อยู่ที่ 80%
- แต่เมื่อมองเป็นภาพรวมการให้กู้ทั้งหมด ธนาคาร A อาจจะมีอัตราการให้กู้อยู่ที่ 80% ส่วน ธนาคาร B มีอัตราการให้กู้อยู่ที่ 70%

ซึ่งกลับเป็นผลลัพธ์เป็นละทางกันเมื่อมองออกมาเป็นภาพรวม\
สำหรับการตัดสินใจว่าจะดูแบบภาพรวมหรือแยกเป็นส่วนอาจจะควรดูจากข้อมูลที่เราสนใจและตัวแปรที่ใช้ตัดสิน
- ถ้า**ตัวแปรที่ใช้ตัดสินใจมีการส่งผลโดยตรงต่อผลลัพธ์** การคำนวณจาก**กลุ่มย่อย**อาจจะเหมาะสมกว่า เนื่องจากข้อมูลที่ใช้ตัดสินใจมีความเกี่ยวเนื่องโดยตรงที่ทำให้เกิดผลลัพธ์ เช่น ในสถาการณ์การกู้เงินอาจจะเป็นเรื่องของกำไรในแต่ละปี เพราะจะส่งผลต่อความเป็นไปได้ในการใช้หนี้กลับอย่างเห็นได้ชัด
- ถ้า**ตัวแปรที่ใช้ตัดสินใจไม่ได้ส่งผลต่อผลลัพธ์จริง ๆ** แต่อาจจะมีตัวแปรอื่นที่มีคั่นกลางการเกิดผลลัพธ์อยู่ การคำนวณแบบ**ภาพรวม**อาจจะมีความเหมาะสมกว่า เนื่องจากเมื่อข้อมูลไม่ได้ส่งผลโดยตรง อาจจะหมายถึงว่าสิ่งที่สนใจไม่ได้เป็นตัวกำหนดผลลัพธ์ ซึ่งทำให้ผลลัพธ์มีความคลาดเคลื่อนได้ เช่น ในสถานการณ์การกู้เงินอาจจะเป็นเรื่องของขนาดบริษัท เพราะขนาดของบริษัทไม่ได้เป็นเหตุที่จะตัดสินความเสี่ยงในการจ่ายหนี้ในอนาคต